**Install and Imports**

In [ ]:
!pip uninstall -y datasets
!pip install datasets==2.18.0 transformers seqeval

Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.4 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=9ca3525a9ec469563ccb0bf79137df37f4b6cf9eb608973291ec8713cb6d60b4
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following de

In [15]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
import numpy as np
from seqeval.metrics import classification_report

**Task 1: Dataset Selection**

In [16]:
dataset = load_dataset("conll2003")

label_list = dataset["train"].features["chunk_tags"].feature.names
print("Labels:", label_list)

/usr/local/lib/python3.12/dist-packages/datasets/load.py:1461: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Labels: ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


**Task 2: Data Preprocessing**

In [17]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

**Task 3: Model Setup**

In [18]:

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [19]:

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)
#Task 4: Training
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no"
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

trainer.train()


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

Step,Training Loss
50,1.794141
100,0.757113
150,0.414608
200,0.356859
250,0.311999
300,0.316756
350,0.307329
400,0.257308
450,0.255918
500,0.246557


TrainOutput(global_step=3512, training_loss=0.21845836197460974, metrics={'train_runtime': 729.324, 'train_samples_per_second': 38.504, 'train_steps_per_second': 4.815, 'total_flos': 1713145849856400.0, 'train_loss': 0.21845836197460974, 'epoch': 2.0})

**Task 5: Evaluation**

In [20]:
predictions, labels, _ = trainer.predict(tokenized_dataset["validation"])
preds = np.argmax(predictions, axis=2)

true_labels = [
    [label_list[l] for l in label if l != -100]
    for label in labels
]

true_preds = [
    [label_list[p] for (p, l) in zip(pred, label) if l != -100]
    for pred, label in zip(preds, labels)
]

print("\nClassification Report:\n")
print(classification_report(true_labels, true_preds))



Classification Report:



/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

        ADJP       0.72      0.62      0.67       306
        ADVP       0.80      0.78      0.79       650
       CONJP       0.60      0.30      0.40        10
        INTJ       0.00      0.00      0.00        31
         LST       0.00      0.00      0.00         3
          NP       0.92      0.91      0.91     14644
          PP       0.97      0.98      0.97      4884
         PRT       0.71      0.78      0.74       147
        SBAR       0.84      0.81      0.83       366
          VP       0.92      0.91      0.91      4696

   micro avg       0.92      0.91      0.92     25737
   macro avg       0.65      0.61      0.62     25737
weighted avg       0.92      0.91      0.92     25737



Task 6: Inference

In [22]:
import torch

# move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

sentence = "John works at Google in California"
tokens = sentence.split()

inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

# move inputs to same device
inputs = {key: val.to(device) for key, val in inputs.items()}

# prediction
outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=2)

predicted_labels = [label_list[p.item()] for p in predictions[0]]

print("\nInference Result:")
for token, label in zip(tokens, predicted_labels):
    print(f"{token} -> {label}")


Inference Result:
John -> O
works -> B-NP
at -> B-VP
Google -> B-PP
in -> B-NP
California -> B-PP


** Task 7: Comparison between POS Tagging and Chunking**
### POS Tagging
- POS (Part-of-Speech) tagging assigns grammatical labels to each word.
- Examples: Noun (NN), Verb (VB), Adjective (JJ)
- It is a **word-level classification task**.
- Helps understand the grammatical structure of a sentence.
- Example:
  - John → Noun
  - runs → Verb

### Chunking
- Chunking groups words into meaningful phrases such as noun phrases (NP) and verb phrases (VP).
- It is a **phrase-level classification task**.
- Helps in identifying sentence structure beyond individual words.
- Example:
  - "John works" → NP + VP

### Key Differences
| Aspect | POS Tagging | Chunking |
|------|------------|----------|
| Level | Word-level | Phrase-level |
| Complexity | Easier | More complex |
| Output | Grammar tags | Phrase groups |
| Use case | Syntax understanding | Structure understanding |

### Conclusion
POS tagging is simpler and focuses on individual words, whereas chunking is more advanced and focuses on grouping words into meaningful phrases.

## Task 8: Report / Blog

### Differences between POS Tagging and Chunking

Part-of-Speech (POS) Tagging and Chunking are both important tasks in Natural Language Processing, but they operate at different levels.

- **POS Tagging** assigns grammatical labels (such as noun, verb, adjective) to each individual word in a sentence. It is a word-level classification task and helps in understanding the syntactic role of each word.

- **Chunking**, also known as shallow parsing, groups words into meaningful phrases such as noun phrases (NP), verb phrases (VP), etc. It is a phrase-level task and provides a higher-level understanding of sentence structure.

**Key Difference:**  
POS tagging focuses on individual words, while chunking focuses on grouping words into phrases.

---

### Challenges Faced

- One of the main challenges was **label alignment** due to BERT’s subword tokenization.
- Words are often split into multiple tokens, so assigning correct labels required careful handling.
- Special tokens like `[CLS]` and `[SEP]` needed to be ignored using `-100`.
- Managing variable sequence lengths and padding was also challenging.
- Some rare labels had low prediction performance due to limited training samples.

---

### Observations and Insights

- The BERT model performed very effectively for token classification tasks.
- Achieved a high F1 score (~0.92), indicating strong model performance.
- Contextual embeddings helped the model understand relationships between words.
- Proper preprocessing (especially label alignment) is critical for good results.
- Even with limited epochs, the model achieved strong accuracy.

---

### Conclusion

This assignment demonstrated that transformer models like BERT are highly powerful for sequence labeling tasks such as POS tagging and chunking. With proper preprocessing and training, high accuracy can be achieved in NLP applications.